# M&A Screening Case File (Project P3)

Quantitative screen across three decision lenses for an e-commerce / logistics
portfolio, run end-to-end on the curated 12-deal book in `data/deals.py`:

1. **The deal book** — 12 public e-commerce / logistics deals (2015–2025).
2. **Accretion / dilution** — 100-unit year-1 EPS sweep (EV/EBITDA × synergy).
3. **DCF discount-rate sensitivity** — WACC 6–12%, value range + implied multiple.
4. **LBO-lite roll-up IRR** — entry × exit multiple at 3/4/5×, 5-yr hold.
5. **EDGAR sanity check** — cached TTM revenue / net-income for key issuers.

All markdown tables are produced live by the code cells below.

In [ ]:
import sys
from pathlib import Path

# Resolve the project root so `model` and `data` import regardless of cwd.
# Works both in a Jupyter kernel (no __file__) and the headless runner,
# which execs the cells with __file__ set to this notebook.
try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    here = Path.cwd()
    ROOT = next((p for p in [here, *here.parents]
                 if (p / "model").is_dir() and (p / "data").is_dir()), here)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from model import accretive, sensitivity, rollup
from data import deals
from data import edgar_sanity

try:
    import tabulate
except ModuleNotFoundError:      # keep the notebook runnable without tabulate
    tabulate = None

def md(df: pd.DataFrame) -> str:
    """Render a DataFrame as a markdown pipe table (tabulate, with fallback)."""
    if tabulate is not None:
        return tabulate.tabulate(df, headers="keys", tablefmt="pipe")
    lines = ["\t".join([str(df.index.name or "")] + [str(c) for c in df.columns])]
    lines.append("\t".join(["----"] * (df.shape[1] + 1)))
    for idx, row in df.iterrows():
        lines.append(str(idx) + "\t" + "\t".join(str(v) for v in row))
    return "\n".join(lines)

## 01 · The deal book
Twelve public e-commerce / logistics transactions. Deal values are announced
figures (USD B); EBITDA multiples are only shown where a clean public multiple
exists — most of these deals are strategic / revenue-priced, so the multiple
column is intentionally sparse (accuracy over completeness).

In [ ]:
deal_rows = []
for i, d in enumerate(deals.deals(), 1):
    mult = (f"{d.ebitda_multiple_of}x ({d.multiple_confidence})"
            if d.ebitda_multiple_of is not None else "n/a")
    deal_rows.append({
        "#": i, "Acquirer": d.acquirer, "Target": d.target, "Year": d.year,
        "Deal ($B)": round(d.deal_value_billion_usd, 2), "Sector": d.target_sector,
        "EV/EBITDA": mult,
    })
deal_df = pd.DataFrame(deal_rows).set_index("#")
print(md(deal_df))
print(f"\n**Avg deal value: ${deals.avg_deal_value():.2f}B**  |  "
      f"{len(deal_df)} deals | documented multiples: {len(deals.deals_with_multiple())}")

## 02 · EPS accretion / dilution (100-unit model)
Year-1 EPS effect of buying a 100-unit EBITDA target at each multiple, with
synergy captured at 0/25/50/100%. A **positive** cell = accretive.
The single most important number for a CV bullet is the top-left (cheapest,
full-capture) cell.

In [ ]:
grid = accretive.accretion_grid()
pct = (grid * 100).round(2)
print("**Year-1 EPS accretion/dilution (%)** — EV/EBITDA multiple × synergy capture\n")
print(md(pct))
best = accretive.best_cell(grid)
print(f"\n**Most accretive cell: {best['multiple']} entry at {best['capture']} "
      f"→ {best['accretion_pct']*100:+.1f}% EPS.** Grid spans "
      f"{best['grid_min_pct']*100:+.1f}% to {best['grid_max_pct']*100:+.1f}%. "
      f"All cells accretive: {best['all_accretive']}")

## 03 · DCF discount-rate sensitivity
Value and implied multiple of a 100-unit EBITDA target as WACC runs 6\u219212%.
The wide value range is the argument for why the entry multiple — not the
terminal rate alone — is what a screen must lock down first.

In [ ]:
wdf = sensitivity.wacc_sweep()
wshow = wdf.copy()
wshow.index = [f"{w*100:.0f}%" for w in wshow.index]
wshow["DCF value"] = wshow["DCF value"].round(1)
wshow["Implied EV/EBITDA"] = wshow["Implied EV/EBITDA"].round(2)
wshow["terminal % of value"] = (wshow["terminal % of value"]*100).round(1)
print("**WACC sweep \u2014 DCF value & implied multiple**\n")
print(md(wshow))
rng = sensitivity.value_range()
print(f"\n**Value range: {rng['value_at_low_wacc']:.0f} \u2192 {rng['value_at_high_wacc']:.0f} "
      f"as WACC goes {rng['wacc_lo']*100:.0f}% \u2192 {rng['wacc_hi']*100:.0f}% "
      f"(a {rng['range_width_pct']:.0f}% spread).**")

## 04 · LBO-lite roll-up IRR matrix
Sponsor IRR by entry multiple (rows) \u00d7 exit multiple (cols) at a stated
leverage, 5-yr hold, 4% EBITDA CAGR. IRR solved by bisection on NPV=0
(no `numpy_financial` dependency). Three leverage ladders shown.

In [ ]:
for lev in rollup.DEFAULT_LEVERAGE:
    m = rollup.rollup_irr_matrix(leverage=lev)
    print(f"**Sponsor IRR (%) \u2014 {lev:.0f}x leverage, 5-yr hold, 4% EBITDA CAGR**\n")
    print(md(m.round(1)))
    print()

## 05 · EDGAR sanity check (cached)
TTM revenue / net income for a few issuers, pulled from SEC EDGAR and cached
to `cache/` so repeat runs do not re-hit the API. This is an order-of-magnitude
cross-check on the acquirer scales the screen assumes — **not** a source of
the deal values (those come from public announcements).

In [ ]:
rows = []
for tk in ("AMZN", "WMT", "UPS"):
    r = edgar_sanity.sanity_check_ticker(tk, use_cache=True)
    rows.append({
        "Ticker": r["ticker"],
        "TTM revenue": (f"${r['revenue']/1e9:,.0f}B" if r["revenue"] is not None else "n/a"),
        "TTM net income": (f"${r['net_income']/1e9:,.0f}B" if r["net_income"] is not None else "n/a"),
        "Source": r["source"] if r["ok"] else "cached/",
    })
print(md(pd.DataFrame(rows).set_index("Ticker")))
print("\n> If a row shows a cache error, set `EDGAR_IDENTITY` and re-run with "
      "`use_cache=False` once; subsequent runs read the JSON cache.")

## 06 · M&A Decision Tool (C1 flagship)

The case file's three screens become a *decision*: synergy-decay (accretive for
N years, then inverts), the accretion zero-line (the "buy only here" boundary),
per-deal IRR waterfalls, and the buy / build / partner trigger line for every
deal in the book. All live in the `decision/` package; this section renders
the exhibits and writes `out/decision_numbers.json` so the numbers are
one-command reproducible.

In [ ]:
import json
import os

from decision import synergy_decay, surfaces, irr_waterfall, decision_page

# --- 06a · Synergy decay: accretive for N years, then inverts --------------
decay = synergy_decay.decay_table()
print("**Synergy decay — year-by-year EPS accretion & years-then-inverts**\n")
show = decay.copy()
for c in show.columns:
    if c != "accretive_years":
        show[c] = (show[c] * 100).round(2)
print(md(show))
print()

# --- 06b · Accretion zero-line (the "buy only here" boundary) -------------
zl = surfaces.zero_line()
print("**Accretion zero-line — entry multiple at which EPS effect = 0**\n")
print(zl.round(3).to_string(), "\n")

trig = surfaces.deal_trigger(deal_price_multiple=8.0, wacc=0.08)
print(f"Deal priced at 8x, 8% WACC: in-zone captures = "
      f"{trig['in_buy_zone_captures']}, min capture = "
      f"{trig['min_capture_in_zone']}. Intrinsic multiple at 8% = "
      f"{trig['intrinsic_multiple_at_wacc']}x.\n")

# --- 06c · Per-deal IRR waterfalls (gate split) ---------------------------
wf = irr_waterfall.per_deal_irr_matrix()
print("**Per-deal IRR waterfalls — 3 scenarios x 12 deals** (first 6)\n")
print(md(wf.head(6)))
print(f"\nAll {len(wf)} IRR cells agree with the gate bridge: "
      f"{bool(wf['IRR agrees w/ bridge'].all())}\n")

# --- 06d · Buy / build / partner decision page ---------------------------
dp = decision_page.page()
print("**Decision page — buy / build / partner trigger per deal**\n")
cols = ["Deal", "Sector", "Recommendation", "Buy cap (x)", "Alt past cap",
        "PV buy", "PV build", "PV partner"]
print(md(dp[cols]))
print()

# --- one-command artifact: the decision numbers ---------------------------
os.makedirs("out", exist_ok=True)
decision_out = {
    "synergy_decay_table": decay.to_dict(),
    "zero_line": {k: round(float(v), 4) for k, v in zl.items()},
    "deal_trigger_8x_8pct": trig,
    "irr_waterfall_matrix": wf.to_dict(),
    "decision_page": dp.to_dict(),
    "headline": {
        "zero_line_0pct_capture": round(float(zl.iloc[0]), 3),
        "zero_line_100pct_capture": round(float(zl.iloc[-1]), 3),
        "intrinsic_multiple_8pct": float(trig["intrinsic_multiple_at_wacc"]),
        "buy_zone_8x_8pct_min_capture": trig["min_capture_in_zone"],
        "n_irr_cells": int(len(wf)),
        "irr_all_agree_with_bridge": bool(wf["IRR agrees w/ bridge"].all()),
        "buy_caps_by_sector": {
            s: dp[dp["Sector"] == s]["Buy cap (x)"].iloc[0]
            for s in dp["Sector"].unique()
        },
    },
}
with open(os.path.join("out", "decision_numbers.json"), "w") as fh:
    json.dump(decision_out, fh, indent=2, default=str)
print("Decision-tool numbers written to out/decision_numbers.json")